<a href="https://colab.research.google.com/github/frazeui/Fraud-Detection-AI-Agent/blob/main/fraud_detection_agent_project1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install fastapi uvicorn nest_asyncio pyngrok

In [ ]:
!pip install langchain-groq langgraph-checkpoint-sqlite

  Using cached langchain_groq-1.1.3-py3-none-any.whl.metadata (2.9 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.8/40.8 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.4/163.4 kB 12.4 MB/s eta 0:00:00


In [ ]:
import os
from typing import Annotated ,TypedDict
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage,SystemMessage
from langchain_groq import ChatGroq
from langgraph.graph import StateGraph,END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode
from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.types import Command,interrupt
import sqlite3
from tenacity import retry,wait_random_exponential,stop_after_attempt


In [ ]:
user_profiles={
    "user_101":{"home_country":"UAE","avg_transactions":500},
    "user_102":{"home_country":"Pakistan","avg_transactions":200},
}

In [ ]:
@tool
def check_amount_risk(amount:float,user_id:str)->str:
    """verify that either the amount which the users is withdrawing is it average based or fraud """

    print(f"[Debug] Received user_id: '{user_id}' (type: {type(user_id)})")
    profile=user_profiles.get(user_id)
    if not profile: return "Error: User profile not found "

    avg=profile["avg_transactions"]

    if amount>avg*10:
        return f"High Risk: [{amount}] is {round(amount/avg,1)} higher than user's average"

    elif amount>avg*3:
        return f"Medium Risk: [{amount}] is {round(amount/avg,1)} higher than user's average"

    else:
        return f"Low risk: [{amount}] is wiithin the normal range"

In [ ]:
@tool
def check_velocity(transactions_count_last_hour:int)->str:
    """verify the number of transactions last hour either it's a fraud or not """


    if transactions_count_last_hour>=5:
        return f"High Risk: {transactions_count_last_hour} times which is unusual velocity"

    elif transactions_count_last_hour>=3 and transactions_count_last_hour<5:
        return f"Medium Risk: {transactions_count_last_hour} time which is unusual velocity"

    else:
        return f"Low Risk: {transactions_count_last_hour} times which is normal range"

In [ ]:
@tool
def check_location_mismatch(user_id:str,transactions_country:str)->str:
    """Tell about the country that either it's matched or not"""

    print(f"[Debug] Received user_id: '{user_id}' (type: {type(user_id)})")
    profile=user_profiles.get(user_id)
    if not profile:return "User profile not found"

    home=profile["home_country"]
    if home.lower()!=transactions_country.lower():
        return f"Medium Risk: Transactoin country is [{transactions_country}] and Home country is [{home}]"
    else:
        return f"Low Risk: Transaction country [{transactions_country}] and Home country [{home}] matches "

In [ ]:
tools=[check_amount_risk,check_velocity,check_location_mismatch]

In [ ]:
class AgentState(TypedDict):
    messages:Annotated[list,add_messages]

In [ ]:
from google.colab import userdata

groq_api=userdata.get("groq_api")
llm=ChatGroq(model="llama-3.1-8b-instant",api_key=groq_api)

llm_with_tools=llm.bind_tools(tools)

In [ ]:
SYSTEM_PROMPT = """
You are a fraud detection analyst assistant. You have access to these exact tools:
- check_amount_risk
- check_velocity
- check_location_mismatch

When a user asks for a transaction analysis:
1. IF you need to call tools, ONLY respond with tool calls. Do NOT include any natural language in this step.
2. AFTER you have received the tool outputs, then combine them into a final risk assessment following these rules:
   a. **Quote tool outputs word-for-word** in your response. Do not paraphrase or summarize.
   b. Classify overall risk as LOW, MEDIUM, or HIGH:
      - LOW → no checks flagged risk
      - MEDIUM → 1 check flagged risk
      - HIGH → 2 or more checks flagged risk
   c. Give a clear recommendation: APPROVE, REVIEW, or BLOCK.

Always use the exact tool names as given. Never guess or hallucinate results.
"""

In [ ]:
def human_review(state:AgentState):
    print("Human Review node reached")
    last_message=state["messages"][-1]
    assessment_text=last_message.content

    needs_review="HIGH" in assessment_text.upper() or 'BLOCK' in assessment_text.upper()

    if not needs_review:return {"messages":[]}
    human_decision=interrupt({"question":"This transaction is in higher risk or blocked,plesae confirm it ","ai_assessment":assessment_text})

    confirmation_message=HumanMessage(content=f"[Human Review]: {human_decision}")
    return {"messages":[confirmation_message]}

In [ ]:
@retry(
    wait=wait_random_exponential(min=1,max=20),
    stop=stop_after_attempt(3)
)
def call_llm(state:AgentState):
    response=llm_with_tools.invoke(state["messages"])
    return {"messages":[response]}

tool_node=ToolNode(tools)

def should_continue(state:AgentState):
    last_message=state["messages"][-1]
    if last_message.tool_calls:
        return "tool_node"
    return "human_review"

In [ ]:
graph=StateGraph(AgentState)
graph.add_node("llm",call_llm)
graph.add_node("tool_node",tool_node)
graph.add_node("human_review",human_review)
graph.set_entry_point("llm")
graph.add_conditional_edges("llm",should_continue,{"tool_node":"tool_node","human_review":"human_review",END:END})
graph.add_edge("tool_node","llm")
graph.add_edge("human_review",END)
conn=sqlite3.connect("graphd.db",check_same_thread=False)
memory=SqliteSaver(conn)
app=graph.compile(checkpointer=memory)

In [ ]:
from fastapi import FastAPI
from pydantic import BaseModel

api=FastAPI(title="Fraud Detection Agent")

class Transactions(BaseModel):
    thread_id:str
    description:str

class HumanDecisionRequest(BaseModel):
    thread_id:str
    decision:str

@api.post("/analyze_transactions")
def analyze_transactions(req:Transactions):

    config={"configurable":{"thread_id":req.thread_id}}
    result = app.invoke({
        "messages": [
            SystemMessage(content=SYSTEM_PROMPT),
            HumanMessage(content=req.description)
        ]
    },config=config)


    if "__interrupt__" in result:
        interrupt_data=result["__interrupt__"][0].value
        return {
            "status":"PENDING REQUEST",
            "thread_id":req.thread_id,
            "ai_assessment":interrupt_data["ai_assessment"],
            "message":"High risk detected .Call/human decision with your decision"

        }

@api.post("/human_decision")
def human_decision(req:HumanDecisionRequest):

    config={"configurable":{"thread_id":req.thread_id}}

    result=app.invoke(Command(resume=req.decision),config=config)

    return {
        "status":"COMPLETED",
        "thread_id":req.thread_id,
        "final_result":result["messages"][-1].content
    }


@api.get("/")
def health_check():
    return {"status":"Fraud Detection AI agent API is running...."}

print(f"FastAPI app ready")


FastAPI app ready


In [ ]:
import nest_asyncio
from pyngrok import ngrok
import uvicorn
from google.colab import userdata

nest_asyncio.apply()

ngrok_token=userdata.get("ngrok_token")
ngrok.set_auth_token(ngrok_token)

public_url=ngrok.connect(8000)

print(f"Public URL: {public_url}")
print(f"API docs: {public_url}/docs")

config=uvicorn.Config(api,host="0.0.0.0",port=8000,log_level="info")
server=uvicorn.Server(config)
await server.serve()


Public URL: NgrokTunnel: "https://garment-shadily-whimsical.ngrok-free.dev" -> "http://localhost:8000"
API docs: NgrokTunnel: "https://garment-shadily-whimsical.ngrok-free.dev" -> "http://localhost:8000"/docs


INFO:     Started server process [524]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


INFO:     39.37.26.107:0 - "GET /docs HTTP/1.1" 200 OK
INFO:     39.37.26.107:0 - "GET /openapi.json HTTP/1.1" 200 OK
[Debug] Received user_id: 'user_101' (type: <class 'str'>)
[Debug] Received user_id: 'user_101' (type: <class 'str'>)
Human Review node reached
INFO:     39.37.26.107:0 - "POST /analyze_transactions HTTP/1.1" 200 OK
Human Review node reached
INFO:     39.37.26.107:0 - "POST /human_decision HTTP/1.1" 200 OK


INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [524]


In [ ]:
if __name__=="__main__":
    analyze_transactions("Analyze this transaction :user_id=user_101,amount=10000,transaction_country=Quwait,transaction_count_last_hour=6",thread_id="txn_1001")

TypeError: analyze_transactions() got an unexpected keyword argument 'thread_id'